### ETL: bronze.weather_history -> silver.weather_history_cleaned

In [0]:
import xml.etree.ElementTree as ET
from pyspark.sql.types as T import ArrayType, StructType, StructField, StringType, FloatType, DateType
from pyspark.sql.functions import udf, explode, col
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
#output schema for array column generated by extract_temp_precip function
schema_filtered_measures = ArrayType(StructType([
    StructField("date", StringType()),
    StructField("time", StringType()),
    StructField("temperature", FloatType()),
    StructField("precipitation", FloatType())
]))

In [0]:

@udf(schema_filtered_measures)
def extract_temp_precip(xml_string):
    """
    Python UDF that parse the XML and search only for specific metrics
    
        Args:
            xml_string: column with string xml
    
        Returns:
            Array column with specific metrics for every date, time
    """
    if not xml_string: return []
    
    try:
        root = ET.fromstring(xml_string)
    except:
        return [] # if and string XML is corrupted
        
    measures = []
    
    for day in root.findall('dia'):
        date = day.get('Dia')
        for time in day.findall('hora'):
            time_val = time.get('Hora')
            meteoros = time.find('Meteoros')
            
            if meteoros is not None:
                temp_val = None
                precip_val = None
                
                # Iteramos sobre todas las métricas de esa hora
                for metric in meteoros:
                    tag = metric.tag
                    
                    # Si empieza con Tem.Aire., guardamos el valor ignorando la altura
                    if tag.startswith('Tem.Aire.'):
                        try: 
                            if metric.text is not None and metric.text.strip() != "":
                                temp_val = float(metric.text)
                        except (ValueError, TypeError): pass
                        
                    # Si empieza con Precip.., guardamos el valor ignorando la altura
                    elif tag.startswith('Precip..'):
                        try: 
                            if metric.text is not None and metric.text.strip() != "":
                                precip_val = float(metric.text)
                        except (ValueError, TypeError): pass
                
                # Si encontró al menos UNA de las dos métricas, agregamos el registro
                # (Si un sensor mide temperatura pero no precipitación, guardará Temp y Null en precipitación)
                if temp_val is not None or precip_val is not None:
                    measures.append((date, time_val, temp_val, precip_val))
                    
    return measures

In [0]:
df_wh_source = spark.sql("select * from dbw_routemind_euskadi_dev.bronze.weather_history limit 3")

In [0]:
df_wh_exploded = df_wh_source.withColumn("extract_data", extract_temp_precip(col("raw_xml"))).select(
        col("sensor_id"),
        explode("extract_data").alias("measure") 
    )

In [0]:

display(df_silver)

In [0]:

df_wh_extracted = df_wh_exploded.select(
    col("sensor_id").alias("sensorId"),
    col("measure.date").cast(T.DateType()).alias("date"),
    col("measure.time").cast(T.StringType()).alias("time"),
    col("measure.temperature").cast(T.FloatType()).alias("temperature"),
    col("measure.precipitation").cast(T.FloatType()).alias("precipitation")
)

In [0]:
notnull_columns = [
    "sensorId",
    "date",
    "time"
]

df_clean = drop_null_required(df_wh_extracted, notnull_columns)

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.weather_history_cleaned"
delta_path = "abfss://silver@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/wheather_history/data"

df_clean.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(target_table)


print(f"APPEND completed on {target_table}. rows processed: {df_clean.count()}")